### HTML aus Ontologie erstellen

In [ ]:
import rdflib
from rdflib.namespace import RDF, RDFS, OWL, DCTERMS, SKOS
from jinja2 import Template
import os

# Pfad zur TTL-Datei
TTL_FILE = "ontology.ttl"
# Ziel-HTML-Datei
OUTPUT_HTML = "../docs/ontology.html"  # GitHub Pages: 'docs/' wird automatisch veröffentlicht

# Lade die Ontologie
g = rdflib.Graph()
g.parse(TTL_FILE, format="ttl")

# Extrahiere Klassen und Properties
classes = []
properties = []

for s in g.subjects(RDF.type, RDFS.Class):
    classes.append({
        "uri": str(s),
        "label": g.value(s, RDFS.label),
        "comment": g.value(s, RDFS.comment),
        "subClassOf": [str(c) for c in g.objects(s, RDFS.subClassOf)]
    })

for s in g.subjects(RDF.type, RDF.Property):
    properties.append({
        "uri": str(s),
        "label": g.value(s, RDFS.label),
        "comment": g.value(s, RDFS.comment),
        "domain": str(g.value(s, RDFS.domain)) if g.value(s, RDFS.domain) else "",
        "range": str(g.value(s, RDFS.range)) if g.value(s, RDFS.range) else "",
        "inverseOf": str(g.value(s, OWL.inverseOf)) if g.value(s, OWL.inverseOf) else "",
        "subPropertyOf": [str(sp) for sp in g.objects(s, RDFS.subPropertyOf)]
    })

# Einfaches HTML Template mit Jinja2
template_str = """
<!DOCTYPE html>
<html lang="de">
<head>
    <meta charset="UTF-8">
    <title>Ontologie Gebietsstammdaten ZH</title>
    <style>
        body { font-family: Arial, sans-serif; margin: 2em; }
        h1 { color: #003366; }
        h2 { color: #0055AA; }
        table { border-collapse: collapse; width: 100%; margin-bottom: 2em; }
        th, td { border: 1px solid #aaa; padding: 0.5em; text-align: left; }
        th { background-color: #eee; }
        a { color: #003366; }
    </style>
</head>
<body>
    <h1>Ontologie: Gebietsstammdaten Zürich</h1>
    <h2>Klassen</h2>
    <table>
        <tr><th>URI</th><th>Label</th><th>Kommentar</th><th>SubClassOf</th></tr>
        {% for c in classes %}
        <tr>
            <td><a href="{{ c.uri }}">{{ c.uri }}</a></td>
            <td>{{ c.label }}</td>
            <td>{{ c.comment }}</td>
            <td>{{ c.subClassOf | join(", ") }}</td>
        </tr>
        {% endfor %}
    </table>

    <h2>Properties</h2>
    <table>
        <tr><th>URI</th><th>Label</th><th>Kommentar</th><th>Domain</th><th>Range</th><th>InverseOf</th><th>SubPropertyOf</th></tr>
        {% for p in properties %}
        <tr>
            <td><a href="{{ p.uri }}">{{ p.uri }}</a></td>
            <td>{{ p.label }}</td>
            <td>{{ p.comment }}</td>
            <td>{{ p.domain }}</td>
            <td>{{ p.range }}</td>
            <td>{{ p.inverseOf }}</td>
            <td>{{ p.subPropertyOf | join(", ") }}</td>
        </tr>
        {% endfor %}
    </table>
</body>
</html>
"""

template = Template(template_str)
html_output = template.render(classes=classes, properties=properties)

# Sicherstellen, dass der Ausgabeordner existiert
os.makedirs(os.path.dirname(OUTPUT_HTML), exist_ok=True)

# HTML speichern
with open(OUTPUT_HTML, "w", encoding="utf-8") as f:
    f.write(html_output)

print(f"HTML-Ontologie erzeugt: {OUTPUT_HTML}")
print("Kann nun via GitHub Pages unter: https://statistikzh.github.io/gebietsstammdaten_zh/ontology.html aufgerufen werden")

ModuleNotFoundError: No module named 'rdf'